<a href="https://colab.research.google.com/github/cols153/posture_checker/blob/main/notebooks/04a_pose_estimation_3D_mmpose_colab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

⚠️ This notebook is intended to run on Google Colab (GPU recommended).

This notebook demonstrates 3D pose estimation using MMPose.  
The model predicts (x, y, z) joint coordinates.  
This notebook is intended for offline demonstration and visualization only.


In [ ]:
# Check that a GPU is available in the Colab runtime
!nvidia-smi

In [ ]:
# Install CondaColab
# The runtime will restart automatically after this cell

!pip -q install -U condacolab
import condacolab
condacolab.install()

In [ ]:
# Create a dedicated Python 3.10 environment for MMPose
!conda create -y -n mmpose310 python=3.10
!conda run -n mmpose310 python -V

In [ ]:
# Install PyTorch with CUDA 12.1 inside the conda environment
!conda run -n mmpose310 python -m pip install -q -U pip setuptools wheel

!conda run -n mmpose310 python -m pip install -q -U \
torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 \
--index-url https://download.pytorch.org/whl/cu121

!conda run -n mmpose310 python -c "import torch; print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())"

In [ ]:
# Install MMEngine and MMCV
!conda run -n mmpose310 python -m pip install -q -U mmengine

!conda run -n mmpose310 python -m pip install -q -U "mmcv==2.2.0" \
-f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

!conda run -n mmpose310 python -c "import mmcv; print('mmcv', mmcv.__version__)"

In [ ]:
# Fix NumPy and OpenCV compatibility issues
!conda run -n mmpose310 python -m pip install -q -U --force-reinstall numpy==1.26.4
!conda run -n mmpose310 python -m pip uninstall -y opencv-python opencv-python-headless
!conda run -n mmpose310 python -m pip install -q opencv-python==4.10.0.84

!conda run -n mmpose310 env MPLBACKEND=Agg python -c "import numpy, cv2; print(numpy.__version__, cv2.__version__)"

In [ ]:
# Install MMDetection and extra dependencies
!conda run -n mmpose310 python -m pip install -q -U xtcocotools munkres
!conda run -n mmpose310 python -m pip install -q -U mmdet

In [ ]:
# Patch MMDetection version check to accept MMCV 2.2.0
!conda run -n mmpose310 bash -lc \
"sed -i \"s/mmcv_maximum_version = '2.2.0'/mmcv_maximum_version = '2.3.0'/\" \
/usr/local/envs/mmpose310/lib/python3.10/site-packages/mmdet/__init__.py"

In [ ]:
# Install MMPose and utility dependencies
!conda run -n mmpose310 python -m pip install -q -U --no-deps mmpose

!conda run -n mmpose310 python -m pip install -q -U \
scipy matplotlib pillow tqdm packaging pyyaml requests einops json_tricks prettytable matplotlib-inline ipython

In [ ]:
# Ensure pkg_resources is available
!conda run -n mmpose310 python -m pip install -q -U "setuptools<80"
!conda run -n mmpose310 python -c "import pkg_resources; print('pkg_resources OK')"

In [ ]:
# Check that MMPose imports correctly
!conda run -n mmpose310 env MPLBACKEND=Agg python -c "import mmpose; print('mmpose OK', mmpose.__version__)"

In [ ]:
# Clone the GitHub repository and define project paths
import os
import sys
import subprocess
from pathlib import Path

REPO_NAME = "posture_checker"
REPO_URL = "https://github.com/cols153/posture_checker.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    notebooks_dir = repo_root / "notebooks"
    os.chdir(notebooks_dir)
else:
    notebooks_dir = Path.cwd()

PROJECT_ROOT = notebooks_dir.parent if notebooks_dir.name == "notebooks" else notebooks_dir

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_IMAGES_DIR = RAW_DIR / "images"
RAW_VIDEOS_DIR = RAW_DIR / "videos"

PROCESSED_DIR = DATA_DIR / "processed"
KEYPOINTS_DIR = PROCESSED_DIR / "keypoints"
FEATURES_DIR = PROCESSED_DIR / "features"

LABELS_DIR = DATA_DIR / "labels"

MODELS_DIR = PROJECT_ROOT / "models"
MMPose_MODELS_DIR = MODELS_DIR / "mmpose"
MLP_MODELS_DIR = MODELS_DIR / "mlp"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
VIS_DIR = OUTPUTS_DIR / "visualizations"
PRED_DIR = OUTPUTS_DIR / "predictions"
LOGS_DIR = OUTPUTS_DIR / "logs"

ALL_DIRS = [
    DATA_DIR,
    RAW_DIR,
    RAW_IMAGES_DIR,
    RAW_VIDEOS_DIR,
    PROCESSED_DIR,
    KEYPOINTS_DIR,
    FEATURES_DIR,
    LABELS_DIR,
    MODELS_DIR,
    MMPose_MODELS_DIR,
    MLP_MODELS_DIR,
    OUTPUTS_DIR,
    VIS_DIR,
    PRED_DIR,
    LOGS_DIR,
]

for d in ALL_DIRS:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())
print("Raw videos dir:", RAW_VIDEOS_DIR)
print("Visualizations dir:", VIS_DIR)
print("Predictions dir:", PRED_DIR)

# --- Specific paths for this notebook ---

# Input video
VIDEO_PATH = RAW_VIDEOS_DIR / "demo" / "pushup_video.mp4"

# JSON outputs
POSE2D_JSON_PATH = PROJECT_ROOT / "data" / "interim" / "keypoints_json" / "pose2d" / "pushup_video.json"
BODY3D_JSON_PATH = PROJECT_ROOT / "data" / "interim" / "keypoints_json" / "body3d" / "pushup_video.json"

# Ensure JSON folders exist
POSE2D_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
BODY3D_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)

# Output video
OUTPUT_VIDEO_PATH = OUTPUTS_DIR / "annotated_pushup_mmpose.mp4"

# Debug
print("Video exists:", VIDEO_PATH.exists())
print("2D JSON:", POSE2D_JSON_PATH)
print("3D JSON:", BODY3D_JSON_PATH)

## Project paths


In [ ]:
# Define the input demo video path inside the repository
VIDEO_IN = RAW_VIDEOS_DIR / "demo" / "pushup_video.mp4"

print("Video path:", VIDEO_IN)


In [ ]:
# Verify that the video exists before running inference
print("Video exists:", VIDEO_IN.exists())
!ls -lh "{VIDEO_IN}"


In [ ]:
# Verify that the video exists before running inference

import os

print("Video exists:", os.path.exists(VIDEO_IN))
!ls -lh "{VIDEO_IN}"

In [ ]:
# Patch the installed MMPose 3D visualizer to replace tostring_rgb() with buffer_rgba()
from pathlib import Path

viz_file = Path("/usr/local/envs/mmpose310/lib/python3.10/site-packages/mmpose/visualization/local_visualizer_3d.py")
text = viz_file.read_text()

old = """        pred_img_data = np.frombuffer(
            fig.canvas.tostring_rgb(), dtype=np.uint8)

        if not pred_img_data.any():
            pred_img_data = np.full((vis_height, vis_width, 3), 255)
        else:
            width, height = fig.get_size_inches() * fig.get_dpi()
            pred_img_data = pred_img_data.reshape(
                int(height),"""

new = """        rgba = np.asarray(fig.canvas.buffer_rgba())
        pred_img_data = np.ascontiguousarray(rgba[..., :3])

        if not pred_img_data.any():
            pred_img_data = np.full((vis_height, vis_width, 3), 255)
        else:
            width, height = fig.get_size_inches() * fig.get_dpi()
            pred_img_data = pred_img_data.reshape(
                int(height),"""

if old in text:
    text = text.replace(old, new)
    viz_file.write_text(text)
    print("Patch applied successfully.")
else:
    print("Exact target block not found.")

In [ ]:
# Run the Body3D pose lifter demo
BODY3D_OUT = OUTPUTS_DIR / "predictions_3d_body3d"
BODY3D_OUT.mkdir(parents=True, exist_ok=True)

!conda run -n mmpose310 python demo/body3d_pose_lifter_demo.py \
    demo/mmdetection_cfg/rtmdet_m_640-8xb32_coco-person.py \
    https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth \
    configs/body_2d_keypoint/rtmpose/body8/rtmpose-m_8xb256-420e_body8-256x192.py \
    https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.pth \
    configs/body_3d_keypoint/motionbert/h36m/motionbert_dstformer-243frm_8xb32-240e_h36m.py \
    https://download.openmmlab.com/mmpose/v1/body_3d_keypoint/pose_lift/h36m/motionbert_ft_h36m-d80af323_20230531.pth \
    --input "{VIDEO_IN}" \
    --output-root "{BODY3D_OUT}" \
    --save-predictions \
    --device cuda


In [ ]:
# Inspect the output folder created by body3d_pose_lifter_demo.py
import os

for root, dirs, files in os.walk(BODY3D_OUT):
    if files:
        print(root, files[:10])

In [ ]:
# Inspect the structure of the Body3D JSON file
print(type(pose3d_body3d))

if isinstance(pose3d_body3d, dict):
    print("Top-level keys:", pose3d_body3d.keys())
    for k, v in pose3d_body3d.items():
        print(f"{k}: type={type(v)}")
        if isinstance(v, list):
            print(f"  length={len(v)}")
elif isinstance(pose3d_body3d, list):
    print("Length:", len(pose3d_body3d))
    print("First element type:", type(pose3d_body3d[0]))
    print("First element:", pose3d_body3d[0])

In [ ]:
# Define proper keypoint extractors for both formats
import numpy as np

def get_kpts_inferencer(frame_record):
    return np.array(frame_record["instances"][0]["keypoints"], dtype=float)

def get_kpts_body3d(frame_record):
    return np.array(frame_record["keypoints"], dtype=float)

In [ ]:
# Inspect the structure of one Body3D frame record
frame_idx = 20

print(type(body3d_frames[frame_idx]))
print(body3d_frames[frame_idx].keys())

for k, v in body3d_frames[frame_idx].items():
    print(f"\n{k}: type={type(v)}")
    if isinstance(v, list):
        print("length:", len(v))
        if len(v) > 0:
            print("first element:", v[0])
    elif isinstance(v, dict):
        print("dict keys:", v.keys())
    else:
        print(v)

In [ ]:
# Extract the frame list from the Body3D JSON
body3d_frames = pose3d_body3d["instance_info"]

print("Inferencer frames:", len(pose3d_inferencer))
print("Body3D frames:", len(body3d_frames))

In [ ]:
# Define keypoint extractors for both JSON formats
import numpy as np

def get_kpts_inferencer(frame_record):
    return np.array(frame_record["instances"][0]["keypoints"], dtype=float)

def get_kpts_body3d(frame_record):
    return np.array(frame_record["instances"][0]["keypoints"], dtype=float)

In [ ]:
# Define a 3D skeleton visualization function
import numpy as np
import matplotlib.pyplot as plt

# COCO-like skeleton (works with MMPose human3d)
SKELETON = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (0, 5), (0, 6), (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 6),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16)
]

def plot_pose3d(ax, kpts, title):
    x, y, z = kpts[:, 0], kpts[:, 1], kpts[:, 2]

    # Scatter joints
    ax.scatter(x, y, z, s=30)

    # Draw skeleton
    for i, j in SKELETON:
        ax.plot([x[i], x[j]],
                [y[i], y[j]],
                [z[i], z[j]])

    ax.set_title(title)

    # Normalize axis for better visualization
    max_range = np.array([
        x.max() - x.min(),
        y.max() - y.min(),
        z.max() - z.min()
    ]).max()

    mid_x = (x.max() + x.min()) / 2
    mid_y = (y.max() + y.min()) / 2
    mid_z = (z.max() + z.min()) / 2

    ax.set_xlim(mid_x - max_range / 2, mid_x + max_range / 2)
    ax.set_ylim(mid_y - max_range / 2, mid_y + max_range / 2)
    ax.set_zlim(mid_z - max_range / 2, mid_z + max_range / 2)

In [ ]:
# Compare the same frame visually for both 3D pipelines
import matplotlib.pyplot as plt

frame_idx = 20

kpts_inf = get_kpts_inferencer(pose3d_inferencer[frame_idx])
kpts_b3d = get_kpts_body3d(body3d_frames[frame_idx])

fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(121, projection="3d")
plot_pose3d(ax1, kpts_inf, f"Inferencer 3D - frame {frame_idx}")

ax2 = fig.add_subplot(122, projection="3d")
plot_pose3d(ax2, kpts_b3d, f"Body3D lifter - frame {frame_idx}")

plt.show()

In [ ]:
# Compare both outputs numerically on one frame using per-joint Euclidean distance
frame_idx = 20

kpts_inf = get_kpts_inferencer(pose3d_inferencer[frame_idx])
kpts_b3d = get_kpts_body3d(body3d_frames[frame_idx])

joint_dist = np.linalg.norm(kpts_inf - kpts_b3d, axis=1)

print("Per-joint distance:", joint_dist)
print("Mean joint distance:", joint_dist.mean())
print("Max joint distance:", joint_dist.max())

In [ ]:
# Compare the mean joint distance across all frames to quantify the difference over time
frame_ids = []
mean_dists = []

n = min(len(pose3d_inferencer), len(body3d_frames))

for i in range(n):
    if len(pose3d_inferencer[i]["instances"]) == 0 or len(body3d_frames[i]["instances"]) == 0:
        continue

    kpts_inf = get_kpts_inferencer(pose3d_inferencer[i])
    kpts_b3d = get_kpts_body3d(body3d_frames[i])

    mean_dist = np.linalg.norm(kpts_inf - kpts_b3d, axis=1).mean()

    frame_ids.append(i)
    mean_dists.append(mean_dist)

plt.figure(figsize=(12, 4))
plt.plot(frame_ids, mean_dists)
plt.title("Mean joint distance between Inferencer and Body3D outputs")
plt.xlabel("frame")
plt.ylabel("mean joint distance")
plt.grid(True, alpha=0.3)
plt.show()

print("Overall mean distance:", np.mean(mean_dists))
print("Overall max distance:", np.max(mean_dists))